<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #0f0c29 0%, #302b63 50%, #24243e 100%); border-radius: 15px; margin: 10px 0;'>
  <h1 style='color: #fff; margin: 0 0 8px 0; font-size: 2.2em;'>🎥 LTX-Video Generator</h1>
  <h3 style='color: #c0c0ff; margin: 0 0 5px 0; font-weight: 400;'>Lightning AI Studio | Wan2GP + mmgp</h3>
  <p style='color: #aaa; margin: 0;'>Text-to-Video & Image-to-Video | Q4_K_M GGUF</p>
</div>

---

### 🚀 Quick Start
1. **GPU Studio → T4** (30 GB RAM)
2. **Cell 1** = setup + download (~10 min)
3. **Cell 2** = launch Gradio

> ⚠️ If `import torch` fails, open a **fresh** T4 instance

In [ ]:
# @title ⚙️ Setup & Download (~17.8 GB)
import os, sys, subprocess
from pathlib import Path

HOME = os.path.expanduser('~')

print('[1/3] Installing dependencies...')
!pip install -q huggingface_hub gradio gguf soundfile

import torch
print(f'  PyTorch {torch.__version__} OK')

print('[2/3] Cloning Wan2GP...')
%cd {HOME}
if not os.path.exists('Wan2GP'):
    !git clone -q https://github.com/deepbeepmeep/Wan2GP.git
%cd Wan2GP
!pip install -q -r requirements.txt
!pip install -q mmgp

print('[3/3] Downloading models...')
from huggingface_hub import hf_hub_download

MODEL_DIR = f'{HOME}/Wan2GP/models'

def dl(repo, filename, dest_dir, dest_name=None):
    Path(dest_dir).mkdir(parents=True, exist_ok=True)
    fp = os.path.join(dest_dir, dest_name or filename)
    if not os.path.exists(fp):
        hf_hub_download(repo_id=repo, filename=filename, local_dir=dest_dir, local_dir_use_symlinks=False)
        tmp = os.path.join(dest_dir, filename)
        if tmp != fp and os.path.exists(tmp):
            os.rename(tmp, fp)
    else:
        print(f'  ✓ {dest_name or filename}')

# GGUF Transformer
dl('Abiray/LTX-2.3-22B-DISTILLED-1.1-GGUF', 'LTX-2.3-22B-distilled-1.1-Q4_K_M.gguf', MODEL_DIR, 'ltx-2.3-22b-distilled-1.1-Q4_K_M.gguf')

# Companion files
dl('DeepBeepMeep/LTX-2', 'ltx-2.3-22b-distilled-lora-384.safetensors', MODEL_DIR)
dl('DeepBeepMeep/LTX-2', 'ltx-2.3-22b_embeddings_connector.safetensors', MODEL_DIR)
dl('DeepBeepMeep/LTX-2', 'ltx-2.3-22b_text_embedding_projection.safetensors', MODEL_DIR)
dl('DeepBeepMeep/LTX-2', 'ltx-2.3-22b_vae.safetensors', MODEL_DIR)
dl('DeepBeepMeep/LTX-2', 'ltx-2.3-22b_audio_vae.safetensors', MODEL_DIR)
dl('DeepBeepMeep/LTX-2', 'ltx-2.3-22b_vocoder.safetensors', MODEL_DIR)
dl('DeepBeepMeep/LTX-2', 'ltx-2.3-spatial-upscaler-x2-1.1.safetensors', MODEL_DIR)
dl('DeepBeepMeep/LTX-2', 'ltx-2.3-temporal-upscaler-x2-1.0.safetensors', MODEL_DIR)

# Gemma text encoder
GEMMA = 'gemma-3-12b-it-qat-q4_0-unquantized'
for gf in ['gemma-3-12b-it-qat-q4_0-unquantized_quanto_bf16_int8.safetensors', 'added_tokens.json', 'chat_template.json', 'config_light.json', 'generation_config.json', 'preprocessor_config.json', 'processor_config.json', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer.model', 'tokenizer_config.json']:
    dl('DeepBeepMeep/LTX-2', f'{GEMMA}/{gf}', f'{MODEL_DIR}/{GEMMA}')

print('✓ Ready! Run Cell 2.')

In [ ]:
# @title 🚀 Launch
import os, sys, gc, json, random, glob, traceback, time
from pathlib import Path
from datetime import datetime

HOME = os.path.expanduser('~')
WAN2GP = f'{HOME}/Wan2GP'
MODEL_DIR = f'{WAN2GP}/models'

os.chdir(WAN2GP)
sys.path.insert(0, WAN2GP)

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import torch
import gradio as gr
from shared.utils.audio_video import save_video

print(f'GPU: {torch.cuda.get_device_name()}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

# ── GGUF Handler ──
import shared.qtypes.gguf

# ── Patch config loading ──
import models.ltx2.ltx2 as ltx2_mod
_original_load = ltx2_mod._load_config_from_checkpoint
def _patched_load(path, fallback_config_path=None):
    from mmgp import quant_router
    if isinstance(path, (list, tuple)):
        path = path[0] if path else ''
    if not path:
        return {}
    try:
        _, metadata = quant_router.load_metadata_state_dict(path)
        if metadata:
            config_raw = metadata.get('config')
            if config_raw:
                config = ltx2_mod._normalize_config(config_raw)
                if config:
                    return config
    except:
        pass
    return {}
ltx2_mod._load_config_from_checkpoint = _patched_load

# ── Load Model ──
print('\nLoading LTX-2.3 22B Distilled 1.1 (Q4_K_M)...')

from mmgp import offload
from shared.utils import files_locator as fl
fl.set_checkpoints_paths(['models', 'ckpts', '.'])
from models.ltx2.ltx2_handler import family_handler

base_model_type = 'ltx2_22B'
model_def = {'ltx2_pipeline': 'distilled'}
extra = family_handler.query_model_def(base_model_type, model_def)
model_def.update(extra)

gemma_folder = f'{MODEL_DIR}/gemma-3-12b-it-qat-q4_0-unquantized'
gemma_files = sorted(glob.glob(f'{gemma_folder}/*.safetensors'))
quanto_files = [f for f in gemma_files if 'quanto' in f]
text_encoder_file = quanto_files[0] if quanto_files else gemma_files[0]

transformer_path = f'{MODEL_DIR}/ltx-2.3-22b-distilled-1.1-Q4_K_M.gguf'

ltx2_model, pipe = family_handler.load_model(
    model_filename=transformer_path,
    model_type='ltx2_22B_distilled',
    base_model_type=base_model_type,
    model_def=model_def,
    dtype=torch.float16,
    VAE_dtype=torch.float16,
    text_encoder_filename=text_encoder_file,
)

# ── mmgp Profile 4 ──
print('Applying mmgp Profile 4...')
offload.profile(
    pipe, profile_no=4, quantizeTransformer=False,
    convertWeightsFloatTo=torch.float16,
    budgets={
        'transformer': 6000, 'text_encoder': 1500,
        'video_encoder': 2000, 'video_decoder': 3000,
        'audio_encoder': 1000, 'audio_decoder': 1000,
        'vocoder': 500, 'spatial_upsampler': 1500,
        'vae': 1000, '*': 1000,
    },
)
offload.shared_state['_attention'] = 'sdpa'
print('✓ Model ready')

# ── Output dir ──
OUTPUT_DIR = Path(f'{HOME}/videos')
OUTPUT_DIR.mkdir(exist_ok=True)

# ── Resolution helper ──
def get_res(base_res, aspect):
    bases = {'1080p': 1088, '720p': 704, '540p': 544, '480p': 480}
    ratios = {'16:9': 16/9, '4:3': 4/3, '1:1': 1.0, '3:4': 3/4, '9:16': 9/16}
    b = bases.get(base_res, 704)
    r = ratios.get(aspect, 16/9)
    if r >= 1.0:
        h, w = b, int(b * r)
    else:
        w, h = b, int(b / r)
    return (w // 32) * 32, (h // 32) * 32

# ── Generate ──
@torch.inference_mode()
def generate(prompt, img_start, img_end, seed, duration, resolution, aspect_ratio, guide_scale, num_steps, progress=gr.Progress()):
    try:
        gc.collect(); torch.cuda.empty_cache()
        progress(0, desc='Starting...')
        
        dur_map = {'2s (49f)': 49, '3s (73f)': 73, '5s (121f)': 121, '8s (193f)': 193, '10s (241f)': 241}
        num_frames = dur_map.get(duration, 121)
        width, height = get_res(resolution, aspect_ratio)
        seed = int(seed) if seed is not None and seed >= 0 else random.randint(0, 2**32-1)
        
        from PIL import Image
        img_s = Image.open(img_start).convert('RGB') if img_start else None
        img_e = Image.open(img_end).convert('RGB') if img_end else None
        
        total_steps = [8]; current_step = [0]; current_pass = [1]
        def cb(step, latent, is_start, override_num_inference_steps=None, pass_no=None, **kwargs):
            if is_start:
                if override_num_inference_steps: total_steps[0] = override_num_inference_steps
                if pass_no: current_pass[0] = pass_no
                current_step[0] = 0
                return
            current_step[0] += 1
            stage = f'Stage {current_pass[0]}'
            frac = current_step[0] / max(total_steps[0], 1)
            if current_pass[0] == 2: frac = 0.73 + 0.22 * frac
            else: frac = frac * 0.73
            progress(min(frac, 0.95), desc=f'{stage}: {current_step[0]}/{total_steps[0]}')
        
        result = ltx2_model.generate(
            input_prompt=prompt, image_start=img_s, image_end=img_e,
            height=height, width=width, frame_num=num_frames,
            fps=24.0, seed=seed, callback=cb,
            VAE_tile_size=256, guide_scale=float(guide_scale),
            sampling_steps=int(num_steps), guide_phases=2,
            n_prompt='', enhance_prompt=False,
        )
        
        progress(0.97, desc='Saving...')
        video_tensor = (result if isinstance(result, dict) else result[0] if isinstance(result, tuple) else result)
        if video_tensor is None:
            return None, 'Error: No video'
        
        video_tensor = video_tensor.cpu()
        ts = datetime.now().strftime('%Y%m%d_%H%M%S')
        out = str(OUTPUT_DIR / f'video_{ts}.mp4')
        
        video_for_save = video_tensor.unsqueeze(0).float() / 127.5 - 1.0
        save_video(tensor=video_for_save, save_file=out, fps=24.0, normalize=True, value_range=(-1, 1))
        
        del video_tensor, video_for_save
        gc.collect(); torch.cuda.empty_cache()
        progress(1.0, desc='Done!')
        return out, f'✓ Seed: {seed} | {width}x{height} | {num_frames}f'
    except Exception as e:
        traceback.print_exc()
        gc.collect(); torch.cuda.empty_cache()
        return None, f'Error: {e}'

# ── Gradio UI ──
with gr.Blocks(title='LTX-Video Generator') as demo:
    gr.Markdown('# 🎥 LTX-Video Generator\nLightning AI T4 | Wan2GP + mmgp | Q4_K_M')
    
    with gr.Tabs():
        with gr.Tab('🎬 Generate'):
            prompt = gr.Textbox(label='Prompt', lines=3, placeholder='A cinematic shot of...')
            with gr.Accordion('🖼️ Image-to-Video (Optional)', open=False):
                with gr.Row():
                    img_start = gr.Image(type='filepath', label='Start Frame')
                    img_end = gr.Image(type='filepath', label='End Frame')
            with gr.Row():
                seed = gr.Number(-1, label='Seed (-1=random)', precision=0)
                duration = gr.Dropdown(['2s (49f)', '3s (73f)', '5s (121f)', '8s (193f)', '10s (241f)'], value='5s (121f)', label='Duration')
            with gr.Row():
                resolution = gr.Dropdown(['1080p', '720p', '540p', '480p'], value='720p', label='Resolution')
                aspect_ratio = gr.Dropdown(['16:9', '4:3', '1:1', '3:4', '9:16'], value='16:9', label='Aspect Ratio')
            with gr.Row():
                guide_scale = gr.Slider(1.0, 8.0, 3.0, step=0.5, label='Guidance Scale')
                num_steps = gr.Slider(2, 8, 8, step=1, label='Steps')
            btn = gr.Button('🎬 Generate', variant='primary')
            video_out = gr.Video(label='Output')
            status = gr.Textbox(label='Status', interactive=False)
            btn.click(generate, [prompt, img_start, img_end, seed, duration, resolution, aspect_ratio, guide_scale, num_steps], [video_out, status])

demo.queue(max_size=3)
demo.launch(share=True)